# 03 · Discovery (Block C)

The consumer now looks for data. Three ways to do that:

| Flow | Who asks whom | Credential filtering |
|---|---|---|
| **C1** direct DSP catalog | consumer → provider | done by the provider, from the consumer's credentials |
| **C2** federated catalog, direct | you → the authority's cache | none |
| **C3** filtered federated catalog | consumer → the authority's catalog filter | by the consumer's credentials |

## Concept: DSP and the catalog

The **Dataspace Protocol (DSP)** is the standard language between connectors. It defines three exchanges: **catalog**, **contract negotiation** and **transfer process**.
Each one is a set of JSON-LD messages sent to the other connector's protocol endpoint (`.../api/dsp/2025-1`).

The catalog exchange is a single request/response: `CatalogRequestMessage` → `Catalog`.
The answer uses **DCAT**, the W3C vocabulary for data catalogs:

```
Catalog (participantId = provider DID)
├── service: DataService   (the provider's DSP endpoint)
└── dataset: Dataset       (one per asset the consumer may see)
    ├── properties          (name, description, DCAT metadata)
    ├── hasPolicy: Offer    (ODRL offer: the contract policy, with an id)
    └── distribution        (how it can be transferred: HttpData-PULL, HttpData-PUSH, AzureStorage-PUSH…)
```

Before answering, the provider authenticates the consumer (DCP token → presentation of its credentials) and evaluates each
contract definition's **access policy** in the `catalog` scope. Datasets whose policy fails are simply left out.

In [1]:
import dataspace as ds

run_id = ds.recall("run_id", produced_by="00_environment.ipynb")
our_assets = {
    "data": ds.recall("data_asset", produced_by="02_publishing.ipynb"),
    "nobody": ds.recall("nobody_asset", produced_by="02_publishing.ipynb"),
    "travel": ds.recall("travel_asset", produced_by="02_publishing.ipynb"),
    "20s": ds.recall("short_lived_asset", produced_by="02_publishing.ipynb"),
}


def dataset_ids(catalog):
    return [dataset["@id"] for dataset in ds.catalog_datasets(catalog)]

## C1 · Direct DSP catalog: the consumer asks the provider

**What:** `POST /consumer/cp/management/v3/catalog/request` on the **consumer's** control plane. We tell our connector *whom* to ask
(`counterPartyAddress` + `counterPartyId`) and *which protocol*. It then sends the DSP `CatalogRequestMessage` itself.

**Returns:** the provider's catalog, as the provider decided this consumer may see it.

In [2]:
catalog_request = {
    "@context": ds.EDC_CONTEXT,
    "@type": "CatalogRequest",
    "counterPartyAddress": ds.dsp_address("provider"),   # http://provider-controlplane:8282/api/dsp/2025-1
    "counterPartyId": ds.did("provider"),
    "protocol": ds.DSP_PROTOCOL,
    "querySpec": {"limit": 500},
}
provider_catalog = ds.post(f"{ds.management_api('consumer')}/catalog/request", catalog_request, expect={200}).json()

visible = dataset_ids(provider_catalog)
print(f"{len(visible)} datasets visible to the consumer")
for label, asset_id in our_assets.items():
    print(f"   {label:7} {asset_id:20} {'visible' if asset_id in visible else 'HIDDEN'}")

POST   /consumer/cp/management/v3/catalog/request  ->  200
52 datasets visible to the consumer
   data    ging-data-1d0d4e     visible
   nobody  ging-nobody-1d0d4e   visible
   travel  ging-travel-1d0d4e   visible
   20s     ging-20s-1d0d4e      visible


What to expect:

- `nobody` is **visible**: `GenericClaim` is not evaluated in the `catalog` scope, only when negotiating.
- `travel` is **hidden**: `RestrictedDiscoveryClaim` fails, because the consumer has no `travel` DomainCredential.
  It shows as visible if the E2E suite already gave the consumer a `VisibilityCredential` (scope `ALL`), which lets it see restricted assets.

### Anatomy of the catalog

In [3]:
print("catalog @type:     ", provider_catalog["@type"])
print("participantId:     ", provider_catalog["participantId"])
print("service:           ", [(s["@type"], s["endpointURL"]) for s in ds.as_list(provider_catalog["service"])])

data_dataset = next(d for d in ds.as_list(provider_catalog["dataset"]) if d["@id"] == our_assets["data"])
print("\ndataset fields:    ", sorted(data_dataset))
print("distributions:     ", [d["format"] for d in ds.as_list(data_dataset["distribution"])])

catalog @type:      Catalog
participantId:      did:web:provider-identityhub%3A8383:api:did
service:            [('DataService', 'http://provider-controlplane:8282/api/dsp/2025-1')]

dataset fields:     ['@id', '@type', 'description', 'distribution', 'edc:contenttype', 'edc:createdAt', 'edc:name', 'hasPolicy', 'id']
distributions:      ['AzureStorage-PUSH', 'HttpData-PULL', 'HttpData-PUSH']


### The offer

`hasPolicy` is an ODRL **Offer**: the contract policy the provider would sign for this dataset. The consumer must send it back **unchanged** to negotiate (notebook 04).

The offer id is not random: it is `base64(contract definition id) : base64(asset id) : base64(uuid)`, so the provider can tell which
contract definition and asset a request refers to.

In [4]:
import base64

offer = ds.as_list(data_dataset["hasPolicy"])[0]
ds.show(offer, max_chars=1200)

print("\noffer id decoded:")
for label, part in zip(["contract definition", "asset", "uuid"], offer["@id"].split(":")):
    print(f"   {label:20} {base64.b64decode(part).decode()}")

{
  "@id": "Y29udHJhY3QtZ2luZy1kYXRhLTFkMGQ0ZQ==:Z2luZy1kYXRhLTFkMGQ0ZQ==:MjZkNmVjNDMtY2JiZi00MmMzLThhMjgtZDA4MDM5OGZhOWEx",
  "@type": "Offer",
  "permission": [
    {
      "action": "use",
      "constraint": [
        {
          "leftOperand": "dse-policy:Membership",
          "operator": "eq",
          "rightOperand": "active"
        }
      ]
    },
    {
      "action": "use",
      "constraint": [
        {
          "leftOperand": "dse-policy:GenericClaim.$.MembershipCredential.name",
          "operator": "eq",
          "rightOperand": "consumer"
        }
      ]
    }
  ]
}

offer id decoded:
   contract definition  contract-ging-data-1d0d4e
   asset                ging-data-1d0d4e
   uuid                 26d6ec43-cbbf-42c3-8a28-d080398fa9a1


## Concept: the federated catalog

Asking every provider one by one does not scale. The authority runs a **federated catalog**:

- a **crawler** that, every 10 seconds, sends a DSP catalog request to every *holder* registered in the issuer (notebook 01) and stores the answers;
- a **cache** you can query with filters, sorting and pagination;
- a **filter service** (DSE) that removes what a given participant may not see, based on its credentials.

The cache is a copy, not the source of truth. New assets appear after the next crawl.

## C2 · Query the cache directly

**What:** `POST /authority/catalog/v1alpha/catalog/query` with a JSON-LD **QuerySpec**. A bare `{}` is rejected because the body cannot be expanded.

**Returns:** an array of catalogs, one per crawled participant.

In [5]:
FEDERATED_CATALOG = f"{ds.BASE_URL}/authority/catalog/v1alpha/catalog/query"


def federated_datasets(query):
    catalogs = ds.post(FEDERATED_CATALOG, query, quiet=True).json()
    datasets = []
    for catalog in catalogs:
        datasets += ds.catalog_datasets(catalog)
    return datasets


everything = {"@context": ds.EDC_CONTEXT, "@type": "QuerySpec", "limit": 500}


def our_data_asset_is_cached():
    return our_assets["data"] in [d["@id"] for d in federated_datasets(everything)]


ds.wait_until("our data asset is in the federated catalog", our_data_asset_is_cached, timeout=60)
print(len(federated_datasets(everything)), "datasets cached in total")

   our data asset is in the federated catalog: yes, after 0s
52 datasets cached in total


Filters work on dataset properties, written as **full JSON-LD IRIs** (`properties.'https://w3id.org/edc/v0.0.1/ns/description'`), because that is how the cache stores them after crawling.
Operators include `=`, `like` (with `%` wildcards) and `in`.

In [6]:
description_contains_consumer = {
    "@context": ds.EDC_CONTEXT,
    "@type": "QuerySpec",
    "filterExpression": [
        {"operandLeft": f"properties.'{ds.EDC}description'", "operator": "like", "operandRight": "%consumer%"}
    ],
    "sortField": f"properties.'{ds.EDC}name'",
    "sortOrder": "ASC",
    "limit": 5,
}
for dataset in federated_datasets(description_contains_consumer):
    print("  ", dataset["@id"])

   ging-data-1d0d4e
   ging-data-2994ed
   ging-data-2bd706
   ging-data-42930a
   ging-data-564922


## C3 · The federated catalog as the consumer is allowed to see it

**What:** `POST /consumer/cp/management/v1alpha/catalog/query` on the consumer's **own** control plane (a DSE endpoint).

**What happens behind it:**
1. the control plane resolves the authority's DID to find the `FederatedCatalogFilterService` endpoint;
2. it gets a token from its wallet (scope "read my credentials", audience = the authority);
3. the filter service fetches the consumer's credentials with that token, queries the cache, and drops datasets whose `RestrictedDiscoveryClaim` fails.

**Returns:** one catalog per crawled participant, already filtered. The keys may come prefixed (`dcat:dataset`, `dspace:participantId`)
depending on how the JSON-LD was compacted; `ds.catalog_datasets` and `ds.catalog_owner` read either form.

In [7]:
FILTERED_CATALOG = f"{ds.BASE_URL}/consumer/cp/management/v1alpha/catalog/query"
catalogs = ds.post(FILTERED_CATALOG, {}, expect={200}).json()

for catalog in catalogs:
    owner = ds.catalog_owner(catalog)
    ids = dataset_ids(catalog)
    ours = sorted(i for i in ids if i.endswith(run_id))
    print(f"catalog of {owner}\n   {len(ids)} datasets, ours: {ours}")

POST   /consumer/cp/management/v1alpha/catalog/query  ->  200
catalog of did:web:provider-identityhub%3A8383:api:did
   50 datasets, ours: ['ging-20s-1d0d4e', 'ging-data-1d0d4e', 'ging-nobody-1d0d4e', 'ging-travel-1d0d4e']
catalog of did:web:consumer-identityhub%3A8383:api:did
   0 datasets, ours: []


Next: **04_negotiation.ipynb**